In [15]:
import pandas as pd
import sqlite3
import requests
from pathlib import Path

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "clairinelou/walmart-hurricane-project/main/data"
)

transactions = pd.read_csv(f"{BASE_URL}/walmart_transactions.csv")

distributions = pd.read_csv(f"{BASE_URL}/distribution_centers.csv")

locations = pd.read_csv(
    f"{BASE_URL}/store_locations.csv",
    dtype={"zip_code": "string"}
)

product = pd.read_csv(f"{BASE_URL}/product_catalog.csv")

response = requests.get(
    f"{BASE_URL}/inventory_and_sales.db",
    timeout=60
)
response.raise_for_status()

db_path = Path("inventory_and_sales.db")
db_path.write_bytes(response.content)

with sqlite3.connect(db_path) as connection:
    daily_sales = pd.read_sql_query(
        "SELECT * FROM daily_sales_summary",
        connection
    )

    inventory = pd.read_sql_query(
        "SELECT * FROM inventory_levels",
        connection
    )

datasets = {
    "transactions": transactions,
    "distributions": distributions,
    "locations": locations,
    "product": product,
    "daily_sales": daily_sales,
    "inventory": inventory
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

transactions: (25562, 8)
distributions: (5, 8)
locations: (14, 8)
product: (45, 7)
daily_sales: (3600, 5)
inventory: (4320, 6)


### Product Catalog Data Cleaning

In [68]:
print('rows:', len(product), '| unique SKUs:', product['sku'].nunique())
print('exact duplicate rows:', product.duplicated().sum())
print('\nmissing values:\n', product.isnull().sum())
print('\nraw category labels:\n', product['category'].value_counts())

rows: 45 | unique SKUs: 45
exact duplicate rows: 0

missing values:
 sku              0
product_name     0
category         0
subcategory     13
unit_cost        5
supplier         0
package_size     0
dtype: int64

raw category labels:
 category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64


See what parts of the data needs cleaning. For example, there's too many SKU's for the number of unique products. There's also missing values in `subcategory` and `unit_cost`

In [69]:
import re
PACK_WORDS = {'pk', 'pack', 'ct'}

def name_key(s):
    s = re.sub(r'[^a-z\s]', ' ', s.lower())
    words = [w[:-1] if w.endswith('s') and len(w) > 3 else w for w in s.split()]
    return ' '.join(sorted(w for w in words if w not in PACK_WORDS))

keys = product['product_name'].map(name_key)
candidates = product.groupby(keys)['sku'].agg(list)
print('SKUs grouped automatically:', candidates[candidates.str.len() > 1].str.len().sum(), 'of', len(product))
print('\nSKUs left on their own (check these by hand):')
print(product.loc[keys.map(candidates.str.len()) == 1, ['sku', 'product_name']].to_string(index=False))

SKUs grouped automatically: 34 of 45

SKUs left on their own (check these by hand):
          sku           product_name
     WTR_24PK        Water - 24 Pack
    STREAM-01   Roku Streaming Stick
     ROKU-STD       Streaming Device
    CHRG-PORT       PORTABLE CHARGER
  PORTCHARGER Portable Phone Charger
        TP-12      TOILET PAPER 12PK
TOILETPAPER12   Toilet Paper 12-Roll
      DIAP-S3  Baby Diapers - Size 3
    DIAPERS-3           DIAPERS SZ 3
     BFORM-01         Infant Formula
  BABYFORMULA           Baby Formula


Group together products that are the same

In [70]:
PRODUCT_FAMILIES = [
    # (canonical_sku,   clean product name,      category,        every SKU that means this product)
    ('POPTART-STRAW', 'Pop-Tarts Strawberry',   'Breakfast',     ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT_STRAWBERRY']),
    ('WATER24PK',     'Bottled Water 24-Pack',  'Emergency',     ['WTR-24', 'WATER24PK', 'WTR_24PK']),
    ('BAT-AA4',       'AA Batteries 4-Pack',    'Emergency',     ['BATT-100', 'BAT-AA4', 'BATT_AA_4PK']),
    ('LED-FLASH',     'LED Flashlight',         'Emergency',     ['FLASH-01', 'LED-FLASH']),
    ('CANDLE-EMRG',   'Emergency Candles',      'Emergency',     ['CNDL-EM', 'CANDLE-EMRG']),
    ('TARP-HD',       'Heavy Duty Tarp',        'Emergency',     ['TARP-HD', 'HDTARP-01']),
    ('BEER-24',       'Light Beer 24-Pack',     'Alcohol',       ['BEER-24', 'LTBEER24']),
    ('MONOPOLY-01',   'Monopoly Board Game',    'Entertainment', ['BG-MONO', 'MONOPOLY-01', 'GAME-MONO']),
    ('PLAYCARD-STD',  'Playing Cards',          'Entertainment', ['CARDS-01', 'PLAYCARD-STD']),
    ('ROKU-STD',      'Roku Streaming Stick',   'Electronics',   ['STREAM-01', 'ROKU-STD']),
    ('PORTCHARGER',   'Portable Phone Charger', 'Electronics',   ['CHRG-PORT', 'PORTCHARGER']),
    ('SOUP-CAN',      'Canned Soup',            'Grocery',       ['SOUP-CAN', 'CNSOUP-01']),
    ('BREAD-WH',      'White Bread',            'Grocery',       ['BREAD-WH', 'WHITEBREAD']),
    ('PB-16',         'Peanut Butter',          'Grocery',       ['PB-16', 'PNUTBTR-16']),
    ('CHIPVARIETY',   'Chips Variety Pack',     'Snacks',        ['CHIPS-VP', 'CHIPVARIETY']),
    ('FIRSTAID-KIT',  'First Aid Kit',          'Health',        ['FAID-01', 'FIRSTAID-KIT']),
    ('PAPERTOWEL6',   'Paper Towels',           'Household',     ['PTOWEL-6', 'PAPERTOWEL6']),
    ('TOILETPAPER12', 'Toilet Paper 12-Roll',   'Household',     ['TP-12', 'TOILETPAPER12']),
    ('DIAPERS-3',     'Diapers Size 3',         'Baby',          ['DIAP-S3', 'DIAPERS-3']),
    ('BABYFORMULA',   'Baby Formula',           'Baby',          ['BFORM-01', 'BABYFORMULA']),
]

sku_map = pd.DataFrame(
    [(variant, canon, name, cat)
     for canon, name, cat, variants in PRODUCT_FAMILIES
     for variant in variants],
    columns=['sku', 'canonical_sku', 'product_name_clean', 'category_clean'],
)

# Every SKU appears exactly once, and every catalog SKU is covered
assert sku_map['sku'].is_unique, 'a SKU is mapped to two products'
assert set(sku_map['sku']) == set(product['sku']), 'mapping and catalog SKUs differ'
sku_map

,sku,canonical_sku,product_name_clean,category_clean
0,PT-12,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast
1,SKU#459812,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast
2,POPTART-STRAW,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast
3,PT_STRAWBERRY,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast
4,WTR-24,WATER24PK,Bottled Water 24-Pack,Emergency
5,WATER24PK,WATER24PK,Bottled Water 24-Pack,Emergency
6,WTR_24PK,WATER24PK,Bottled Water 24-Pack,Emergency
7,BATT-100,BAT-AA4,AA Batteries 4-Pack,Emergency
8,BAT-AA4,BAT-AA4,AA Batteries 4-Pack,Emergency
9,BATT_AA_4PK,BAT-AA4,AA Batteries 4-Pack,Emergency


In [71]:
CATEGORY_LABELS = {
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'entertainment': 'Entertainment',
    'electronics': 'Electronics',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'snacks': 'Snacks',
    'household': 'Household',
    'health': 'Health',
    'baby': 'Baby', 'baby products': 'Baby',
}

In [72]:
product["sku"] = (
    product["sku"].astype("string").str.strip().str.upper()
)

In [73]:
def clean_products(df, sku_map):
    out = df.copy()   # never modify the raw frame

    # 1. Category labels: strip + lowercase, then map to the 10 database names
    out['category_from_label'] = out['category'].str.strip().str.lower().map(CATEGORY_LABELS)
    assert out['category_from_label'].notna().all(), 'a category label is not in CATEGORY_LABELS'

    # 2. Attach canonical SKU, clean name and category
    out = out.merge(sku_map, on='sku', how='left', validate='one_to_one')

    # Cross-check: the cleaned label must agree with the mapping table's category
    mismatch = out[out['category_from_label'] != out['category_clean']]
    assert mismatch.empty, f'category disagreement:\n{mismatch}'
    out = out.drop(columns='category_from_label')

    # 3. unit_cost: flag missing values, do not fill them
    out['unit_cost_missing'] = out['unit_cost'].isna()

    # 4. Drop columns we will not use (see decision log)
    out = out.drop(columns=['subcategory', 'supplier', 'package_size'])
    return out

# Ensure sku_map is present and cleanly construct product_clean
product_clean = clean_products(product, sku_map)

assert len(product_clean) == len(product), 'row count changed'
print('rows:', len(product), '->', len(product_clean))
print('canonical products:', product_clean['canonical_sku'].nunique())
print('\ncategories after cleaning:\n', product_clean['category_clean'].value_counts())
display(product_clean.head(10))

rows: 45 -> 45
canonical products: 20

categories after cleaning:
 category_clean
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


,sku,product_name,category,unit_cost,canonical_sku,product_name_clean,category_clean,unit_cost_missing
0,PT-12,pop-tarts strawberry,Breakfast Foods,1.93,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,False
1,SKU#459812,Strawberry Pop-Tart,Breakfast,2.26,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,False
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.11,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,False
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,NaN,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,True
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,2.64,WATER24PK,Bottled Water 24-Pack,Emergency,False
5,WATER24PK,Bottled Water 24-Pack,Emergency,2.38,WATER24PK,Bottled Water 24-Pack,Emergency,False
6,WTR_24PK,Water - 24 Pack,Emergency,2.08,WATER24PK,Bottled Water 24-Pack,Emergency,False
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,NaN,BAT-AA4,AA Batteries 4-Pack,Emergency,True
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,4.16,BAT-AA4,AA Batteries 4-Pack,Emergency,False
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,3.71,BAT-AA4,AA Batteries 4-Pack,Emergency,False


In [74]:
# SKUs whose product family has more than one listed pack size
pack_size_differs = (
    product.merge(sku_map, on="sku")
    .groupby("canonical_sku")["package_size"]
    .transform("nunique") > 1
)

product_decision_log = pd.DataFrame([
    {
        "Decision": "Standardize category labels",
        "Reason": "27 spellings of 10 categories; matched the database names so we can join to daily_sales_summary for Q5.",
        "Rows affected": (product_clean["category"] != product_clean["category_clean"]).sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Map 45 SKUs to 20 canonical products",
        "Reason": "Fragmented SKUs hide true demand (the Q2 Pop-Tarts effect). Name-matching found 34; the other 11 were matched by hand.",
        "Rows affected": (product_clean["sku"] != product_clean["canonical_sku"]).sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Merge different pack sizes into one product",
        "Reason": "package_size contradicts the product names (Bottled Water 24PK is listed as '1 pack'), so it can't be trusted to split products.",
        "Rows affected": pack_size_differs.sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Keep board games and playing cards as separate products",
        "Reason": "Q5 has to test whether the +250% board-game surge is really playing cards lumped in.",
        "Rows affected": product_clean["category_clean"].eq("Entertainment").sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Flag missing unit_cost, don't fill it",
        "Reason": "unit_cost_missing flags missing unit_cost values without touching the raw data.",
        "Rows affected": product_clean["unit_cost_missing"].sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Drop supplier, subcategory and package_size columns",
        "Reason": "Suppliers are wrong (Hasbro supplies beer), subcategories are cut off ('First', 'Peanut'), pack sizes contradict names.",
        "Rows affected": len(product),
        "Rows removed": 0
    }
])

display(product_decision_log)

,Decision,Reason,Rows affected,Rows removed
0,Standardize category labels,27 spellings of 10 categories; matched the dat...,32,0
1,Map 45 SKUs to 20 canonical products,Fragmented SKUs hide true demand (the Q2 Pop-T...,25,0
2,Merge different pack sizes into one product,package_size contradicts the product names (Bo...,37,0
3,Keep board games and playing cards as separate...,Q5 has to test whether the +250% board-game su...,5,0
4,"Flag missing unit_cost, don't fill it",unit_cost_missing flags missing unit_cost valu...,5,0
5,"Drop supplier, subcategory and package_size co...","Suppliers are wrong (Hasbro supplies beer), su...",45,0


### Transactions Data Cleaning

In [24]:
transactions = pd.read_csv(f"{BASE_URL}/walmart_transactions.csv")
transactions.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82


In [25]:
print(transactions.shape)
display(transactions.dtypes)
transactions.info()
display(transactions.describe(include="all"))
display(transactions.isnull().sum())

print("Duplicate rows:", transactions.duplicated().sum())

(25562, 8)


,0
transaction_id,object
store_id,object
timestamp,object
sku,object
product_name,object
category,object
quantity,int64
unit_price,object


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25562 entries, 0 to 25561
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaction_id  25562 non-null  object
 1   store_id        25562 non-null  object
 2   timestamp       24676 non-null  object
 3   sku             25562 non-null  object
 4   product_name    25562 non-null  object
 5   category        25562 non-null  object
 6   quantity        25562 non-null  int64 
 7   unit_price      25562 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.6+ MB


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
count,25562,25562,24676,25562,25562,25562,25562.000000,25562
unique,25532,60,23952,45,63,37,NaN,3668
top,TXN004028,FL-445,09/24/2024 21:09,TARP-HD,LED Flashlight,EMERGENCY,NaN,3.39
freq,2,1819,5,766,518,1823,NaN,94
mean,NaN,NaN,NaN,NaN,NaN,NaN,1.932634,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,1.086926,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,-5.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,NaN


,0
transaction_id,0
store_id,0
timestamp,886
sku,0
product_name,0
category,0
quantity,0
unit_price,0


Duplicate rows: 30


In [26]:
def remove_duplicates(df):
    return df.drop_duplicates().copy()


def clean_text_and_ids(df):
    text_columns = [
        "transaction_id", "store_id", "timestamp",
        "sku", "product_name", "category", "unit_price"
    ]

    for col in text_columns:
        df[col] = df[col].str.strip()

    # Convert blank strings to missing values
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Standardize store IDs
    df["store_id"] = (
        df["store_id"]
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
    )

    df["sku"] = df["sku"].str.upper()

    return df


def clean_timestamps(df):
    df["timestamp_original"] = df["timestamp"]

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed",
        dayfirst=False,
        errors="coerce"
    )

    return df


def clean_prices(df):
    df["unit_price"] = pd.to_numeric(
        df["unit_price"]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    return df


def clean_categories(df):
    category_mapping = {
        "emergency supplies": "emergency",
        "groceries": "grocery",
        "games & entertainment": "entertainment",
        "household essentials": "household",
        "baby products": "baby",
        "breakfast foods": "breakfast",
        "alcoholic beverages": "alcohol",
        "health & wellness": "health"
    }

    df["category"] = (
        df["category"].str.lower().replace(category_mapping)
    )

    return df


def clean_quantities(df):
    df["quantity_invalid"] = df["quantity"] < 0

    df["quantity"] = (
        df["quantity"]
        .mask(df["quantity_invalid"], pd.NA)
        .astype("Int64")
    )

    return df

In [27]:
def consolidate_skus(df):
    out = df.merge(
        sku_map,
        on="sku",
        how="left",
        validate="many_to_one"
    )

    assert len(out) == len(df), "Row count changed"
    assert out["canonical_sku"].notna().all(), "Some SKUs did not match"

    return out

In [28]:
def clean_transactions(df):
    cleaned = df.copy()

    cleaned = remove_duplicates(cleaned)
    cleaned = clean_text_and_ids(cleaned)
    cleaned = consolidate_skus(cleaned)
    cleaned = clean_timestamps(cleaned)
    cleaned = clean_prices(cleaned)
    cleaned = clean_categories(cleaned)
    cleaned = clean_quantities(cleaned)

    return cleaned


# Apply all cleaning steps to the original data
transactions_clean = clean_transactions(transactions)

In [29]:
transactions_clean["timestamp_missing"] = (
    transactions_clean["timestamp"].isna()
)

transactions_clean["quantity_missing"] = (
    transactions_clean["quantity"].isna()
)

In [30]:
transactions_clean.info()
display(transactions_clean.isnull().sum())

print(
    "Repeated transaction IDs:",
    transactions_clean["transaction_id"].duplicated().sum()
)

display(transactions_clean["store_id"].unique())
display(transactions_clean["category"].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25532 entries, 0 to 25531
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   transaction_id      25532 non-null  object        
 1   store_id            25532 non-null  object        
 2   timestamp           24648 non-null  datetime64[ns]
 3   sku                 25532 non-null  object        
 4   product_name        25532 non-null  object        
 5   category            25532 non-null  object        
 6   quantity            25402 non-null  Int64         
 7   unit_price          25532 non-null  float64       
 8   canonical_sku       25532 non-null  object        
 9   product_name_clean  25532 non-null  object        
 10  category_clean      25532 non-null  object        
 11  timestamp_original  24648 non-null  object        
 12  quantity_invalid    25532 non-null  bool          
 13  timestamp_missing   25532 non-null  bool      

,0
transaction_id,0
store_id,0
timestamp,884
sku,0
product_name,0
category,0
quantity,130
unit_price,0
canonical_sku,0
product_name_clean,0


Repeated transaction IDs: 0


array(['FL-710', 'FL-412', 'FL-105', 'FL-330', 'FL-302', 'FL-445',
       'FL-619', 'FL-287', 'FL-239', 'FL-156', 'FL-501', 'FL-078'],
      dtype=object)

array(['grocery', 'household', 'electronics', 'alcohol', 'snacks',
       'emergency', 'entertainment', 'health', 'breakfast', 'baby'],
      dtype=object)

In [31]:
print("Original SKUs:", transactions_clean["sku"].nunique())
print("Canonical products:", transactions_clean["canonical_sku"].nunique())

display(
    transactions_clean[["sku", "canonical_sku"]]
    .drop_duplicates()
    .sort_values(["canonical_sku", "sku"])
)

Original SKUs: 45
Canonical products: 20


,sku,canonical_sku
49,BABYFORMULA,BABYFORMULA
30,BFORM-01,BABYFORMULA
85,BAT-AA4,BAT-AA4
38,BATT-100,BAT-AA4
28,BATT_AA_4PK,BAT-AA4
4,BEER-24,BEER-24
22,LTBEER24,BEER-24
128,BREAD-WH,BREAD-WH
15,WHITEBREAD,BREAD-WH
9,CANDLE-EMRG,CANDLE-EMRG


In [32]:
step1 = remove_duplicates(transactions.copy())
step2 = clean_text_and_ids(step1.copy())
step3 = consolidate_skus(step2.copy())
step4 = clean_timestamps(step3.copy())
step5 = clean_prices(step4.copy())
step6 = clean_categories(step5.copy())
step7 = clean_quantities(step6.copy())

text_changed = (
    step1.fillna("").astype(str)
    != step2.fillna("").astype(str)
).any(axis=1)

transaction_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicates",
        "Reason": "Avoid counting the same transaction twice.",
        "Rows affected": len(transactions) - len(step1),
        "Rows removed": len(transactions) - len(step1)
    },
    {
        "Decision": "Clean text and standardize IDs",
        "Reason": "Remove extra spaces, handle blanks, and make IDs consistent.",
        "Rows affected": int(text_changed.sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Apply shared SKU mapping",
        "Reason": "Attach canonical SKUs, clean names, and categories.",
        "Rows affected": int(step3["canonical_sku"].notna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Parse timestamps",
        "Reason": "Make dates usable for daily sales and weather analysis.",
        "Rows affected": int(step4["timestamp"].notna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Keep missing timestamps",
        "Reason": "The correct dates are unknown.",
        "Rows affected": int(step4["timestamp"].isna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Convert prices to numeric",
        "Reason": "Remove dollar signs and commas to allow calculations.",
        "Rows affected": int(step5["unit_price"].notna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Normalize category labels",
        "Reason": "Combine equivalent labels into consistent categories.",
        "Rows affected": int(
            step5["category"].ne(step6["category"]).sum()
        ),
        "Rows removed": 0
    },
    {
        "Decision": "Replace negative quantities with missing values",
        "Reason": "The correct quantities cannot be verified.",
        "Rows affected": int(step7["quantity_invalid"].sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Flag missing timestamps and quantities",
        "Reason": "Identify records unsuitable for analyses requiring these fields.",
        "Rows affected": int(
            (
                step7["timestamp"].isna()
                | step7["quantity"].isna()
            ).sum()
        ),
        "Rows removed": 0
    }
])

display(transaction_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicates,Avoid counting the same transaction twice.,30,30
1,Clean text and standardize IDs,"Remove extra spaces, handle blanks, and make I...",5725,0
2,Apply shared SKU mapping,"Attach canonical SKUs, clean names, and catego...",25532,0
3,Parse timestamps,Make dates usable for daily sales and weather ...,24648,0
4,Keep missing timestamps,The correct dates are unknown.,884,0
5,Convert prices to numeric,Remove dollar signs and commas to allow calcul...,25532,0
6,Normalize category labels,Combine equivalent labels into consistent cate...,19112,0
7,Replace negative quantities with missing values,The correct quantities cannot be verified.,130,0
8,Flag missing timestamps and quantities,Identify records unsuitable for analyses requi...,1011,0


I cleaned the transaction data by removing 30 duplicate rows, standardizing text, store IDs, and category labels, converting timestamps and prices to appropriate data types, and applying our shared SKU mapping. I replaced 130 negative quantities with missing values because the correct quantities were unknown and left 884 missing timestamps unchanged. I preserved the original data, added flags for missing or invalid values, and organized the cleaning steps into a reusable pipeline.


### Distribution Data Cleaning

In [33]:
distributions = pd.read_csv(f"{BASE_URL}/distribution_centers.csv")
distributions.head()

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,~750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


In [34]:
distributions.shape

(5, 8)

In [35]:
distributions.dtypes

,0
dc_id,object
dc_name,object
city,object
state,object
avg_daily_shipments,int64
surge_capacity,object
distance_to_orlando_mi,int64
cost_per_shipment,float64


In [36]:
distributions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dc_id                   5 non-null      object 
 1   dc_name                 5 non-null      object 
 2   city                    5 non-null      object 
 3   state                   5 non-null      object 
 4   avg_daily_shipments     5 non-null      int64  
 5   surge_capacity          5 non-null      object 
 6   distance_to_orlando_mi  5 non-null      int64  
 7   cost_per_shipment       4 non-null      float64
dtypes: float64(1), int64(2), object(5)
memory usage: 452.0+ bytes


In [37]:
distributions.describe(include="all")

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
count,5,5,5,5,5.00000,5,5.000000,4.000000
unique,5,5,5,4,NaN,5,NaN,NaN
top,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,NaN,1000,NaN,NaN
freq,1,1,1,2,NaN,1,NaN,NaN
mean,NaN,NaN,NaN,NaN,360.00000,NaN,485.000000,410.625000
std,NaN,NaN,NaN,NaN,96.17692,NaN,377.987434,197.064023
min,NaN,NaN,NaN,NaN,250.00000,NaN,141.000000,245.000000
25%,NaN,NaN,NaN,NaN,300.00000,NaN,235.000000,284.750000
50%,NaN,NaN,NaN,NaN,350.00000,NaN,438.000000,355.250000
75%,NaN,NaN,NaN,NaN,400.00000,NaN,503.000000,481.125000


In [38]:
distributions.isnull().sum()

,0
dc_id,0
dc_name,0
city,0
state,0
avg_daily_shipments,0
surge_capacity,0
distance_to_orlando_mi,0
cost_per_shipment,1


In [39]:
print("Duplicate rows:", distributions.duplicated().sum())
print("Duplicate DC IDs:", distributions["dc_id"].duplicated().sum())

Duplicate rows: 0
Duplicate DC IDs: 0


In [40]:
# Keep the original data
distributions_clean = distributions.copy()

# Remove duplicate rows
distributions_clean = distributions_clean.drop_duplicates()

# Remove extra spaces from text columns
for col in ["dc_id", "dc_name", "city", "state"]:
    distributions_clean[col] = distributions_clean[col].str.strip()

# Standardize capitalization
distributions_clean["dc_id"] = distributions_clean["dc_id"].str.upper()
distributions_clean["state"] = distributions_clean["state"].str.upper()

# Flag approximate capacities before removing "~"
distributions_clean["surge_capacity_approximate"] = (
    distributions_clean["surge_capacity"].astype(str).str.contains("~")
)

# Remove "~" and convert the column to numeric
distributions_clean["surge_capacity"] = pd.to_numeric(
    distributions_clean["surge_capacity"]
    .astype(str)
    .str.replace("~", "", regex=False)
    .str.strip()
)

In [41]:
display(distributions_clean)
distributions_clean.info()
distributions_clean.isnull().sum()

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment,surge_capacity_approximate
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0,False
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5,False
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0,False
3,DC-MIA,Miami Distribution Center,Miami,FL,350,750,235,298.0,True
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   dc_id                       5 non-null      object 
 1   dc_name                     5 non-null      object 
 2   city                        5 non-null      object 
 3   state                       5 non-null      object 
 4   avg_daily_shipments         5 non-null      int64  
 5   surge_capacity              5 non-null      int64  
 6   distance_to_orlando_mi      5 non-null      int64  
 7   cost_per_shipment           4 non-null      float64
 8   surge_capacity_approximate  5 non-null      bool   
dtypes: bool(1), float64(1), int64(3), object(4)
memory usage: 457.0+ bytes


,0
dc_id,0
dc_name,0
city,0
state,0
avg_daily_shipments,0
surge_capacity,0
distance_to_orlando_mi,0
cost_per_shipment,1
surge_capacity_approximate,0


In [42]:
distribution_text_cols = ["dc_id", "dc_name", "city", "state"]

spaces_changed = pd.Series(False, index=distributions.index)

for col in distribution_text_cols:
    spaces_changed |= distributions[col].ne(
        distributions[col].str.strip()
    )

capitalization_changed = pd.Series(False, index=distributions.index)

for col in ["dc_id", "state"]:
    stripped = distributions[col].str.strip()
    capitalization_changed |= stripped.ne(stripped.str.upper())

distribution_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicates",
        "Reason": "Avoid repeated distribution center records.",
        "Rows affected": int(distributions.duplicated().sum()),
        "Rows removed": int(distributions.duplicated().sum())
    },
    {
        "Decision": "Remove extra spaces",
        "Reason": "Keep text formatting consistent.",
        "Rows affected": int(spaces_changed.sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize ID and state capitalization",
        "Reason": "Use consistent uppercase labels.",
        "Rows affected": int(capitalization_changed.sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Convert surge capacity to numeric",
        "Reason": "Allow capacity calculations.",
        "Rows affected": len(distributions_clean),
        "Rows removed": 0
    },
    {
        "Decision": "Preserve approximate capacity with a flag",
        "Reason": "Record that Miami's capacity is approximate after removing ~.",
        "Rows affected": int(
            distributions_clean["surge_capacity_approximate"].sum()
        ),
        "Rows removed": 0
    },
    {
        "Decision": "Keep missing shipment cost",
        "Reason": "Birmingham's actual cost is unknown.",
        "Rows affected": int(
            distributions_clean["cost_per_shipment"].isna().sum()
        ),
        "Rows removed": 0
    }
])

display(distribution_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicates,Avoid repeated distribution center records.,0,0
1,Remove extra spaces,Keep text formatting consistent.,0,0
2,Standardize ID and state capitalization,Use consistent uppercase labels.,0,0
3,Convert surge capacity to numeric,Allow capacity calculations.,5,0
4,Preserve approximate capacity with a flag,Record that Miami's capacity is approximate af...,1,0
5,Keep missing shipment cost,Birmingham's actual cost is unknown.,1,0


I cleaned the distribution center data by removing extra spaces and standardizing ID and state capitalization. I converted surge capacity to numeric values and added a flag to preserve that Miami’s capacity was approximate. I left Birmingham’s shipment cost missing because its actual cost was unknown. No duplicate rows were found.

### Locations Data Cleaning

In [43]:
locations = pd.read_csv(
    f"{BASE_URL}/store_locations.csv",
    dtype={"zip_code": "string"}
)
locations.head()

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


In [44]:
print(locations.shape)
display(locations.dtypes)
locations.info()
display(locations.describe(include="all"))
display(locations.isnull().sum())
display(locations)

(14, 8)


,0
store_id,object
store_name,object
city,object
state,object
zip_code,string[python]
latitude,float64
longitude,float64
region,object


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   store_id    14 non-null     object 
 1   store_name  14 non-null     object 
 2   city        14 non-null     object 
 3   state       14 non-null     object 
 4   zip_code    14 non-null     string 
 5   latitude    13 non-null     float64
 6   longitude   13 non-null     float64
 7   region      14 non-null     object 
dtypes: float64(2), object(5), string(1)
memory usage: 1.0+ KB


,store_id,store_name,city,state,zip_code,latitude,longitude,region
count,14,14,14,14,14,13.000000,13.000000,14
unique,14,12,12,3,12,NaN,NaN,7
top,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,NaN,NaN,Central
freq,1,2,2,10,2,NaN,NaN,5
mean,NaN,NaN,NaN,NaN,NaN,28.418192,-82.069631,NaN
std,NaN,NaN,NaN,NaN,NaN,1.319349,1.676108,NaN
min,NaN,NaN,NaN,NaN,NaN,25.761700,-87.216900,NaN
25%,NaN,NaN,NaN,NaN,NaN,27.950600,-82.324800,NaN
50%,NaN,NaN,NaN,NaN,NaN,28.292000,-81.655700,NaN
75%,NaN,NaN,NaN,NaN,NaN,29.210800,-81.379200,NaN


,0
store_id,0
store_name,0
city,0
state,0
zip_code,0
latitude,1
longitude,1
region,0


,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central
5,FL-156,Walmart Supercenter Daytona Beach,Daytona Beach,FL,32114,29.2108,-81.0228,East
6,FL-501,Walmart Supercenter Fort Myers,Fort Myers,FL,33901,26.6406,-81.8723,West
7,FL-078,Walmart Supercenter Gainesville,Gainesville,FL,32601,29.6516,-82.3248,North
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle
9,FL-445,Walmart Supercenter St. Petersburg,St. Petersburg,Fla.,33701,27.7676,-82.6403,West


In [45]:
def clean_location_text(df):
    for col in ["store_id", "store_name", "city", "state", "region"]:
        df[col] = df[col].str.strip()

    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Use consistent state and region labels
    df["state"] = (
        df["state"]
        .str.upper()
        .replace({"FLORIDA": "FL", "FLA.": "FL"})
    )

    df["region"] = df["region"].str.title()

    return df


def clean_location_ids(df):
    # Match the format used in transactions_clean
    df["store_id"] = (
        df["store_id"]
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
    )

    return df


def clean_location_types(df):
    # ZIP codes are identifiers, so store them as text
    df["zip_code"] = df["zip_code"].astype("string").str.zfill(5)

    for col in ["latitude", "longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Flag missing coordinates without guessing replacements
    df["coordinates_missing"] = (
        df[["latitude", "longitude"]].isna().any(axis=1)
    )

    return df


def clean_locations(df):
    cleaned = df.copy()

    cleaned = clean_location_text(cleaned)
    cleaned = clean_location_ids(cleaned)
    cleaned = clean_location_types(cleaned)

    # Standardize first so the duplicate stores become identical
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)

    return cleaned

In [46]:
locations_clean = clean_locations(locations)

display(locations_clean)
locations_clean.info()
display(locations_clean.isnull().sum())

print("Rows before:", len(locations))
print("Rows after:", len(locations_clean))
print(
    "Repeated store IDs:",
    locations_clean["store_id"].duplicated().sum()
)

,store_id,store_name,city,state,zip_code,latitude,longitude,region,coordinates_missing
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central,False
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West,False
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South,False
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North,False
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,FL,34741,28.2920,-81.4076,Central,False
5,FL-156,Walmart Supercenter Daytona Beach,Daytona Beach,FL,32114,29.2108,-81.0228,East,False
6,FL-501,Walmart Supercenter Fort Myers,Fort Myers,FL,33901,26.6406,-81.8723,West,False
7,FL-078,Walmart Supercenter Gainesville,Gainesville,FL,32601,29.6516,-82.3248,North,False
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle,True
9,FL-445,Walmart Supercenter St. Petersburg,St. Petersburg,FL,33701,27.7676,-82.6403,West,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   store_id             12 non-null     object 
 1   store_name           12 non-null     object 
 2   city                 12 non-null     object 
 3   state                12 non-null     object 
 4   zip_code             12 non-null     string 
 5   latitude             11 non-null     float64
 6   longitude            11 non-null     float64
 7   region               12 non-null     object 
 8   coordinates_missing  12 non-null     bool   
dtypes: bool(1), float64(2), object(5), string(1)
memory usage: 912.0+ bytes


,0
store_id,0
store_name,0
city,0
state,0
zip_code,0
latitude,1
longitude,1
region,0
coordinates_missing,0


Rows before: 14
Rows after: 12
Repeated store IDs: 0


In [47]:
location_text_cols = [
    "store_id", "store_name", "city", "state", "region"
]

spaces_changed = pd.Series(False, index=locations.index)

for col in location_text_cols:
    spaces_changed |= locations[col].ne(locations[col].str.strip())

original_ids = locations["store_id"].str.strip()

standard_ids = (
    original_ids
    .str.upper()
    .str.replace("_", "-", regex=False)
    .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
)

states = locations["state"].str.strip()
standard_states = (
    states.str.upper()
    .replace({"FLORIDA": "FL", "FLA.": "FL"})
)

location_log = pd.DataFrame([
    {
        "Decision": "Remove extra spaces",
        "Reason": "Keep text formatting consistent.",
        "Rows affected": int(spaces_changed.sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize store IDs",
        "Reason": "Match transaction IDs and identify duplicate stores.",
        "Rows affected": int(original_ids.ne(standard_ids).sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize state labels",
        "Reason": "Use FL consistently.",
        "Rows affected": int(states.ne(standard_states).sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize region capitalization",
        "Reason": "Use consistent region labels.",
        "Rows affected": int(
            locations["region"].str.strip().ne(
                locations["region"].str.strip().str.title()
            ).sum()
        ),
        "Rows removed": 0
    },
    {
        "Decision": "Convert ZIP codes to text",
        "Reason": "ZIP codes are identifiers, not measurements.",
        "Rows affected": len(locations),
        "Rows removed": 0
    },
    {
        "Decision": "Remove duplicate stores after standardizing",
        "Reason": "Keep one record per store.",
        "Rows affected": len(locations) - len(locations_clean),
        "Rows removed": len(locations) - len(locations_clean)
    },
    {
        "Decision": "Keep and flag missing coordinates",
        "Reason": "The correct coordinates are unknown.",
        "Rows affected": int(
            locations_clean["coordinates_missing"].sum()
        ),
        "Rows removed": 0
    }
])

display(location_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove extra spaces,Keep text formatting consistent.,1,0
1,Standardize store IDs,Match transaction IDs and identify duplicate s...,2,0
2,Standardize state labels,Use FL consistently.,4,0
3,Standardize region capitalization,Use consistent region labels.,0,0
4,Convert ZIP codes to text,"ZIP codes are identifiers, not measurements.",14,0
5,Remove duplicate stores after standardizing,Keep one record per store.,2,2
6,Keep and flag missing coordinates,The correct coordinates are unknown.,1,0


In [48]:
transactions_with_locations = transactions_clean.merge(
    locations_clean,
    on="store_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows before:", len(transactions_clean))
print("Rows after:", len(transactions_with_locations))

print(
    "Unmatched store IDs:",
    transactions_with_locations.loc[
        transactions_with_locations["_merge"] == "left_only",
        "store_id"
    ].unique()
)

assert len(transactions_with_locations) == len(transactions_clean)

transactions_with_locations = (
    transactions_with_locations.drop(columns="_merge")
)

Rows before: 25532
Rows after: 25532
Unmatched store IDs: []


I standardized store IDs and state labels, removed extra spaces, and converted ZIP codes to text. After standardizing the IDs, I removed duplicate records for Orlando and Kissimmee, leaving 12 unique stores. I kept Tallahassee’s missing coordinates as missing and added a flag so they can be handled explicitly during geographic analysis.

### Inventory and Sales Cleaning

---



In [49]:
display(inventory.head())
display(daily_sales.head())

,date,store_id,sku,units_on_hand,units_on_order,reorder_point
0,2024-09-12,FL-239,CHIPS-VP,112,5.0,31
1,2024-09-12,FL-239,PAPERTOWEL6,106,46.0,36
2,2024-09-12,FL-239,PT-12,62,36.0,30
3,2024-09-12,FL-239,PORTCHARGER,185,62.0,40
4,2024-09-12,FL-239,BFORM-01,88,1.0,35


,date,store_id,category,total_units,total_revenue
0,2024-09-12,FL-239,Emergency,90.0,1582.55
1,2024-09-12,FL-239,Breakfast,57.0,356.99
2,2024-09-12,FL-239,Alcohol,41.0,246.65
3,2024-09-12,FL-239,Entertainment,53.0,916.56
4,2024-09-12,FL-239,Grocery,69.0,339.30


The code above is used to display the first five rows of the loaded inventory dataset that we are given to work with. This allows us to get a small preview of the different columns and raw entries to see what we are working with. It helps us verify the database table is loaded in accurately and see the raw values like the different mixed dates and store IDs before we start the cleaning process.

In [50]:
print("Inventory:", inventory.shape)
inventory.info()

print("Daily sales:", daily_sales.shape)
daily_sales.info()

Inventory: (4320, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4320 entries, 0 to 4319
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            4320 non-null   object 
 1   store_id        4320 non-null   object 
 2   sku             4320 non-null   object 
 3   units_on_hand   4320 non-null   int64  
 4   units_on_order  3694 non-null   float64
 5   reorder_point   4320 non-null   int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 202.6+ KB
Daily sales: (3600, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           3600 non-null   object 
 1   store_id       3600 non-null   object 
 2   category       3600 non-null   object 
 3   total_units    3600 non-null   float64
 4   total_revenue  3600 non-null   float64
dtypes: floa

The code above is printing the shape, which is the number of rows and columsn in the dataframe for the structural summary for both the inventory and daily sales DataFrames. The summary shows the coliumn data types and counts of the non-ull values which lets us quickly spot any missing information and data which confirms that both datasets are loaded in correctly before we start the cleaning.

In [51]:
import pandas as pd
import numpy as np

# Make a copy of the original sales summary dataframe
sales_clean = daily_sales.copy()

# --- DECISION LOG AND CLEANING PROCESS ---
# 1. Store ID Standardization:
# - Issue: Ensure alignment with standard 'FL-XXX' format used in other tables (handling lowercase or underscores if present).
# - Decision: Strip spaces, convert to upper case, replace underscores with hyphens, and ensure proper padding.
sales_clean['store_id'] = (
    sales_clean['store_id']
    .str.strip()
    .str.upper()
    .str.replace('_', '-', regex=False)
    .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
)

# 2. Date Standardization:
# - Issue: Dates are stored in mixed formats (e.g., '2024-09-12' and integer-like strings '20240911').
# - Decision: Standardize date column by converting to standard datetime objects using pd.to_datetime with format='mixed'.
sales_clean['date'] = pd.to_datetime(
    sales_clean['date'].astype(str).str.strip(),
    format='mixed',
    errors='coerce'
)

# 3. Category Standardization:
# - Issue: Inconsistencies like 'Emergency', 'Breakfast', 'Electronics', 'Snacks' compared to standardized categories in transactions.
# - Decision: Normalize category labels, then use title case to match category_clean.
category_mapping = {
    'emergency': 'emergency',
    'emergency supplies': 'emergency',
    'breakfast': 'breakfast',
    'breakfast foods': 'breakfast',
    'alcohol': 'alcohol',
    'alcoholic beverages': 'alcohol',
    'entertainment': 'entertainment',
    'games & entertainment': 'entertainment',
    'grocery': 'grocery',
    'groceries': 'grocery',
    'electronics': 'electronics',
    'snacks': 'snacks',
    'household': 'household',
    'household essentials': 'household',
    'health': 'health',
    'health & wellness': 'health',
    'baby': 'baby',
    'baby products': 'baby'
}
sales_clean['category'] = sales_clean['category'].str.strip().str.lower().replace(category_mapping)
sales_clean["category"] = sales_clean["category"].str.title()

# 4. Total Units & Total Revenue Check:
# - Decision: Preserve numeric unit totals and flag fractional values without rounding.
temp_units = pd.to_numeric(sales_clean['total_units'], errors='coerce')

sales_clean["total_units"] = temp_units

sales_clean["total_units_fractional"] = (
    temp_units.notna() & temp_units.mod(1).ne(0)
)
sales_clean['total_revenue'] = pd.to_numeric(sales_clean['total_revenue'], errors='coerce')

sales_clean["total_revenue_negative"] = (
    sales_clean["total_revenue"] < 0
)

print(
    "Rows with negative revenue:",
    sales_clean["total_revenue_negative"].sum()
)

# 5. Duplicates Check:
# - Issue: Check for potential duplicate reports for the same date, store, and category.
# - Decision: Drop exact duplicate rows if any exist.
sales_clean = sales_clean.drop_duplicates().reset_index(drop=True)

# Print decision log summary
print("=== DAILY SALES SUMMARY DECISION LOG ===")
print("1. Dates: Standardized mixed formats (e.g., '20240911' to '2024-09-11') into datetime64[ns].")
print("2. Store IDs: Standardized to uppercase hyphenated format (e.g., 'FL-239').")
print("3. Categories: Standardized labels to match category_clean.")
print("4. Units: Preserved numeric totals and flagged fractional values.")
print("5. Revenue: Converted to numeric and flagged negative values.")
print("6. Exact duplicate rows removed:", len(daily_sales) - len(sales_clean))
print("\n--- Cleaned Sales Summary Info ---")
sales_clean.info()
print("\nNulls per column:\n", sales_clean.isnull().sum())
display(sales_clean.head())


Rows with negative revenue: 34
=== DAILY SALES SUMMARY DECISION LOG ===
1. Dates: Standardized mixed formats (e.g., '20240911' to '2024-09-11') into datetime64[ns].
2. Store IDs: Standardized to uppercase hyphenated format (e.g., 'FL-239').
3. Categories: Standardized labels to match category_clean.
4. Units: Preserved numeric totals and flagged fractional values.
5. Revenue: Converted to numeric and flagged negative values.
6. Exact duplicate rows removed: 0

--- Cleaned Sales Summary Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    3600 non-null   datetime64[ns]
 1   store_id                3600 non-null   object        
 2   category                3600 non-null   object        
 3   total_units             3600 non-null   float64       
 4   total_revenue           3600 

,date,store_id,category,total_units,total_revenue,total_units_fractional,total_revenue_negative
0,2024-09-12,FL-239,Emergency,90.0,1582.55,False,False
1,2024-09-12,FL-239,Breakfast,57.0,356.99,False,False
2,2024-09-12,FL-239,Alcohol,41.0,246.65,False,False
3,2024-09-12,FL-239,Entertainment,53.0,916.56,False,False
4,2024-09-12,FL-239,Grocery,69.0,339.30,False,False


The code above is standardizing the text formats, dates and the category labels for the daily sales dataset so it aligns them perfectly with the other cleaned corproate table. This code is bascially conducting an overhual cleaning to ensure everything is in order. This code also checks for and flags any of the data that is out of the ordianry, like any negative revenues or fractional items sold. It does this without changign the raw numbers and concludes the logging for each of the actions. This is the beginning portion of the data cleaning section before getting into the more intricate parts. It sets a over-all cleaned dataset to get into more of the specific cleaning necessary.

In [52]:
sales_clean["total_revenue_negative"] = (
    sales_clean["total_revenue"] < 0
)

# Separate subset for revenue analysis
sales_revenue_valid = sales_clean.loc[
    sales_clean["total_revenue"].notna()
    & ~sales_clean["total_revenue_negative"]
].copy()

The code above is dedicated to block identifying and filtering out any of the out of ordinary negative revenue records that are building a cleaned subset specifically dedicated to sales revenue analysis. By removing the incorrect entries, it allows the subsequent financial calculations and trend models to remain accurate and not be skeqed by any reporting errors.

In [53]:
import pandas as pd

print("=== PRODUCT CATALOG COLUMNS ===\n")
# 1. sku
print("1. 'sku':")
print("Total unique SKUs:", product['sku'].nunique())
print("Sample SKUs (checking format variations):", product['sku'].unique()[:10])
print("Check duplicates/variants:", product[product['sku'].str.upper().duplicated(keep=False)]['sku'].tolist())

# 2. product_name
print("\n2. 'product_name':")
print("Unique product names:", product['product_name'].unique())

# 3. category
print("\n3. 'category':")
print("Unique values:", product['category'].unique())

# 4. subcategory
print("\n4. 'subcategory':")
print("Unique values:", product['subcategory'].unique())
print("Null values:", product['subcategory'].isnull().sum())

# 5. unit_cost
print("\n5. 'unit_cost':")
print("Null values:", product['unit_cost'].isnull().sum())
print("Range of costs:", product['unit_cost'].min(), "to", product['unit_cost'].max())

# 6. supplier
print("\n6. 'supplier':")
print("Unique values:", product['supplier'].unique())

# 7. package_size
print("\n7. 'package_size':")
print("Unique values:", product['package_size'].unique())

=== PRODUCT CATALOG COLUMNS ===

1. 'sku':
Total unique SKUs: 45
Sample SKUs (checking format variations): <StringArray>
[        'PT-12',    'SKU#459812', 'POPTART-STRAW', 'PT_STRAWBERRY',
        'WTR-24',     'WATER24PK',      'WTR_24PK',      'BATT-100',
       'BAT-AA4',   'BATT_AA_4PK']
Length: 10, dtype: string
Check duplicates/variants: []

2. 'product_name':
Unique product names: ['pop-tarts strawberry' 'Strawberry Pop-Tart' 'Pop-Tarts Strawberry'
 'BOTTLED WATER 24PK' 'Bottled Water 24-Pack' 'Water - 24 Pack'
 'BATTERIES AA 4PK' 'AA Batteries 4-Pack' 'FLASHLIGHT LED'
 'Light Beer (24-pack)' 'Monopoly Board Game' 'Board Game - Monopoly'
 'Playing Cards' 'PLAYING CARDS' 'Roku Streaming Stick' 'Streaming Device'
 'Soup - Canned' 'Canned Soup' 'WHITE BREAD' 'PEANUT BUTTER'
 'Peanut Butter' 'Candles (Emergency)' 'Emergency Candles' 'First-Aid Kit'
 'First Aid Kit' 'PORTABLE CHARGER' 'Portable Phone Charger'
 'Variety Pack - Chips' 'Chips Variety Pack' 'Paper Towels'
 'TOILET PAPER

The code above is dedicated to inspecting the indvidual columns of the raw product catalog so it can diagnose formatting issues and any missing information before cleaning it. This prints a sumamry statistics like the unique SKU counts, category spellings, missing values and varying package sizes to help design the clean-up mapping.

In [54]:
print("=== INVENTORY LEVELS COLUMNS ===\n")
# 1. date
print("1. 'date':")
print("Unique values count:", inventory['date'].nunique())
print("Min date:", inventory['date'].min(), "| Max date:", inventory['date'].max())

# 2. store_id
print("\n2. 'store_id':")
print("Unique store IDs:", inventory['store_id'].unique())

# 3. sku
print("\n3. 'sku':")
print("Unique SKU count in inventory:", inventory['sku'].nunique())
print("SKUs that don't match cleaned transaction SKUs:")
cleaned_tx_skus = set(transactions_clean['sku'].unique())
inv_skus = set(inventory['sku'].unique())
print(inv_skus - cleaned_tx_skus)

# 4. units_on_hand
print("\n4. 'units_on_hand':")
print("Min:", inventory['units_on_hand'].min(), "| Max:", inventory['units_on_hand'].max())

# 5. units_on_order
print("\n5. 'units_on_order':")
print("Nulls:", inventory['units_on_order'].isnull().sum())
print("Min:", inventory['units_on_order'].min(), "| Max:", inventory['units_on_order'].max())

# 6. reorder_point
print("\n6. 'reorder_point':")
print("Min:", inventory['reorder_point'].min(), "| Max:", inventory['reorder_point'].max())

=== INVENTORY LEVELS COLUMNS ===

1. 'date':
Unique values count: 30
Min date: 2024-09-01 | Max date: 2024-09-30

2. 'store_id':
Unique store IDs: ['FL-239' 'FL-105' 'FL-330' 'FL-412' 'FL-287' 'FL-156' 'FL-501' 'FL-078'
 'FL-619' 'FL-445' 'FL-302' 'FL-710']

3. 'sku':
Unique SKU count in inventory: 49
SKUs that don't match cleaned transaction SKUs:
{'MISC-001', 'LEGACY-PT', 'OLD-BATT-50', 'UNKNOWN-99'}

4. 'units_on_hand':
Min: 5 | Max: 200

5. 'units_on_order':
Nulls: 626
Min: 0.0 | Max: 100.0

6. 'reorder_point':
Min: 20 | Max: 50


The code above insepcts indvidual columns of the raw invetory dataset so it can identify any formatting issues, invalid dates and any unmatched SKU code before the intense cleaning process. Bascially knowing what to clean before it starts, so we don't clean things that didn't need cleaning. By printing the medtadata counts, coordiante ranges, and listing mistmatched IDs, it helps to establish the stadrdized validation cosntracts for the database import pipeline.

In [55]:
def clean_inventory(df, sku_map):
    out = df.copy()

    # Remove exact duplicate records
    out = out.drop_duplicates()

    # Standardize store IDs
    out["store_id"] = (
        out["store_id"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
    )

    # Standardize SKU formatting
    out["sku"] = (
        out["sku"].astype("string").str.strip().str.upper()
    )

    # Parse dates and flag missing or unparseable dates
    out["date_original"] = out["date"]

    out["date"] = pd.to_datetime(
        out["date"].astype("string").str.strip(),
        format="mixed",
        errors="coerce"
    ).dt.normalize()

    out["date_missing"] = out["date"].isna()

    # Convert inventory quantities to numeric
    numeric_columns = [
        "units_on_hand", "units_on_order", "reorder_point"
    ]

    for col in numeric_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

        # Flag questionable values for investigation
        out[f"{col}_missing"] = out[col].isna()
        out[f"{col}_negative"] = out[col].lt(0)
        out[f"{col}_fractional"] = (
            out[col].notna() & out[col].mod(1).ne(0)
        )

    # Standardization can reveal additional exact duplicates
    out = out.drop_duplicates()

    # Attach the same canonical product information as transactions
    rows_before_mapping = len(out)

    out = out.merge(
        sku_map,
        on="sku",
        how="left",
        validate="many_to_one"
    )

    assert len(out) == rows_before_mapping, "Mapping changed row count"

    out["sku_unmapped"] = out["canonical_sku"].isna()

    # Flag repeated reports without automatically deleting them
    out["duplicate_inventory_key"] = out.duplicated(
        subset=["date", "store_id", "sku"],
        keep=False
    )

    return out.reset_index(drop=True)


inventory_clean = clean_inventory(inventory, sku_map)

print("Rows before:", len(inventory))
print("Rows after:", len(inventory_clean))
print("Exact duplicates removed:", len(inventory) - len(inventory_clean))

display(inventory_clean.head())
inventory_clean.info()

Rows before: 4320
Rows after: 4320
Exact duplicates removed: 0


,date,store_id,sku,units_on_hand,units_on_order,reorder_point,date_original,date_missing,units_on_hand_missing,units_on_hand_negative,...,units_on_order_negative,units_on_order_fractional,reorder_point_missing,reorder_point_negative,reorder_point_fractional,canonical_sku,product_name_clean,category_clean,sku_unmapped,duplicate_inventory_key
0,2024-09-12,FL-239,CHIPS-VP,112,5.0,31,2024-09-12,False,False,False,...,False,False,False,False,False,CHIPVARIETY,Chips Variety Pack,Snacks,False,False
1,2024-09-12,FL-239,PAPERTOWEL6,106,46.0,36,2024-09-12,False,False,False,...,False,False,False,False,False,PAPERTOWEL6,Paper Towels,Household,False,False
2,2024-09-12,FL-239,PT-12,62,36.0,30,2024-09-12,False,False,False,...,False,False,False,False,False,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,False,False
3,2024-09-12,FL-239,PORTCHARGER,185,62.0,40,2024-09-12,False,False,False,...,False,False,False,False,False,PORTCHARGER,Portable Phone Charger,Electronics,False,False
4,2024-09-12,FL-239,BFORM-01,88,1.0,35,2024-09-12,False,False,False,...,False,False,False,False,False,BABYFORMULA,Baby Formula,Baby,False,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4320 entries, 0 to 4319
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       4320 non-null   datetime64[ns]
 1   store_id                   4320 non-null   string        
 2   sku                        4320 non-null   object        
 3   units_on_hand              4320 non-null   int64         
 4   units_on_order             3694 non-null   float64       
 5   reorder_point              4320 non-null   int64         
 6   date_original              4320 non-null   object        
 7   date_missing               4320 non-null   bool          
 8   units_on_hand_missing      4320 non-null   bool          
 9   units_on_hand_negative     4320 non-null   bool          
 10  units_on_hand_fractional   4320 non-null   bool          
 11  units_on_order_missing     4320 non-null   bool          
 12  units_

The code above implements the clean_inventory pipeline to standardize the store IDs, format the SKUs, normalize mixed date formats, and convert inventory metrics into clean numeric values. It also safely identifies and flags any unmapped legacy SKUs, missing values, and duplicate records without prematurely dropping critical rows from the inventory dataset.  

In [56]:
# Count all flags
flag_columns = [
    col for col in inventory_clean.columns
    if col.endswith(("_missing", "_negative", "_fractional"))
    or col in ["sku_unmapped", "duplicate_inventory_key"]
]

display(inventory_clean[flag_columns].sum())

# Display records needing investigation
display(
    inventory_clean.loc[
        inventory_clean[flag_columns].any(axis=1)
    ].head(20)
)

# Check that inventory stores exist in the locations table
unmatched_stores = (
    set(inventory_clean["store_id"].dropna())
    - set(locations_clean["store_id"])
)

print("Inventory store IDs missing from locations:", unmatched_stores)

,0
date_missing,0
units_on_hand_missing,0
units_on_hand_negative,0
units_on_hand_fractional,0
units_on_order_missing,626
units_on_order_negative,0
units_on_order_fractional,0
reorder_point_missing,0
reorder_point_negative,0
reorder_point_fractional,0


,date,store_id,sku,units_on_hand,units_on_order,reorder_point,date_original,date_missing,units_on_hand_missing,units_on_hand_negative,...,units_on_order_negative,units_on_order_fractional,reorder_point_missing,reorder_point_negative,reorder_point_fractional,canonical_sku,product_name_clean,category_clean,sku_unmapped,duplicate_inventory_key
13,2024-09-12,FL-105,LEGACY-PT,176,46.0,33,2024-09-12,False,False,False,...,False,False,False,False,False,NaN,NaN,NaN,True,False
31,2024-09-12,FL-330,LEGACY-PT,115,47.0,37,2024-09-12,False,False,False,...,False,False,False,False,False,NaN,NaN,NaN,True,False
33,2024-09-12,FL-330,CNSOUP-01,122,NaN,21,2024-09-12,False,False,False,...,False,False,False,False,False,SOUP-CAN,Canned Soup,Grocery,False,False
37,2024-09-12,FL-412,PNUTBTR-16,199,NaN,42,2024-09-12,False,False,False,...,False,False,False,False,False,PB-16,Peanut Butter,Grocery,False,False
38,2024-09-12,FL-412,CHRG-PORT,129,NaN,40,2024-09-12,False,False,False,...,False,False,False,False,False,PORTCHARGER,Portable Phone Charger,Electronics,False,False
48,2024-09-12,FL-287,LEGACY-PT,190,22.0,28,2024-09-12,False,False,False,...,False,False,False,False,False,NaN,NaN,NaN,True,False
54,2024-09-12,FL-287,TOILETPAPER12,94,NaN,40,2024-09-12,False,False,False,...,False,False,False,False,False,TOILETPAPER12,Toilet Paper 12-Roll,Household,False,False
55,2024-09-12,FL-287,WTR_24PK,102,NaN,49,2024-09-12,False,False,False,...,False,False,False,False,False,WATER24PK,Bottled Water 24-Pack,Emergency,False,False
58,2024-09-12,FL-287,HDTARP-01,190,NaN,21,2024-09-12,False,False,False,...,False,False,False,False,False,TARP-HD,Heavy Duty Tarp,Emergency,False,False
59,2024-09-12,FL-287,BATT_AA_4PK,61,NaN,22,2024-09-12,False,False,False,...,False,False,False,False,False,BAT-AA4,AA Batteries 4-Pack,Emergency,False,False


Inventory store IDs missing from locations: set()


Inventory cleaning identified 626 missing units-on-order values and 355 records with SKUs absent from the shared product mapping. We preserved these records and flagged the issues rather than guessing quantities or product identities. All store IDs matched the locations table, and no negative quantities, fractional quantities, missing dates, or repeated date/store/SKU combinations were found.

In [57]:
unmapped_skus = (
    inventory_clean.loc[
        inventory_clean["sku_unmapped"], "sku"
    ]
    .value_counts()
    .rename_axis("sku")
    .reset_index(name="rows")
)

display(unmapped_skus)

,sku,rows
0,UNKNOWN-99,91
1,LEGACY-PT,90
2,OLD-BATT-50,90
3,MISC-001,84


The code above is filtering the cleaned inventory dataset so it can isolate the unmapped items and aggregate their occurence counts into a structured sumamry table. By displaying the unmapped SKUs alongside the raw row frequencies it can provide us a clear target for the future catalog addictions and the legacy item investigations.

In [58]:
print("Fractional unit totals:",
      sales_clean["total_units_fractional"].sum())

print("Negative revenue records:",
      sales_clean["total_revenue_negative"].sum())

Fractional unit totals: 76
Negative revenue records: 34


preserved and flagged the 76 fractional unit totals and 34 negative revenues records.

In [59]:
print("Negative unit totals:", sales_clean["total_units"].lt(0).sum())

print("Repeated date/store/category reports:")
display(sales_clean.loc[
    sales_clean.duplicated(
        ["date", "store_id", "category"],
        keep=False
    )
])

Negative unit totals: 0
Repeated date/store/category reports:


,date,store_id,category,total_units,total_revenue,total_units_fractional,total_revenue_negative


Four inventory SKUs were absent from the product catalog: UNKNOWN-99 (91 records), LEGACY-PT (90), OLD-BATT-50 (90), and MISC-001 (84). We retained and flagged these records but excluded them from product-level comparisons requiring a verified mapping.

# Product Catalog Data Decision Log

In [60]:
import pandas as pd

# Step 1: Remove exact duplicates (if any)
prod_step1 = product.drop_duplicates().copy()

# Step 2: Strip and standardize SKU casing
prod_step2 = prod_step1.copy()
prod_step2["sku"] = prod_step2["sku"].astype("string").str.strip().str.upper()

# Step 3: Standardize category labels and verify mapping
prod_step3 = prod_step2.copy()
prod_step3["category_from_label"] = prod_step3["category"].str.strip().str.lower().map(CATEGORY_LABELS)

# Step 4: Map messy raw SKUs to canonical product codes
prod_step4 = prod_step3.merge(sku_map, on="sku", how="left")

# Step 5: Drop unused columns (subcategory, supplier, package_size)
prod_step5 = prod_step4.drop(columns=["subcategory", "supplier", "package_size"])

# Calculate metrics dynamically
sku_changes = (prod_step1["sku"].astype(str) != prod_step2["sku"].astype(str)).sum()
category_changes = (prod_step2["category"] != prod_step4["category_clean"]).sum()
sku_mapped_count = (prod_step4["sku"] != prod_step4["canonical_sku"]).sum()
cost_missing = prod_step4["unit_cost"].isna().sum()

product_decision_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicates",
        "Reason": "Ensure clean starting baseline for catalog lookup.",
        "Rows affected": len(product) - len(prod_step1),
        "Rows removed": len(product) - len(prod_step1)
    },
    {
        "Decision": "Standardize raw SKUs",
        "Reason": "Strip whitespace and uppercase values to allow clean merging with transaction tables.",
        "Rows affected": int(sku_changes),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize category labels",
        "Reason": "Map diverse text strings to official normalized categories compatible with database joins.",
        "Rows affected": int(category_changes),
        "Rows removed": 0
    },
    {
        "Decision": "Map variants to canonical SKUs",
        "Reason": "Consolidate multiple variant SKUs (such as pack sizes) under single corporate parent products.",
        "Rows affected": int(sku_mapped_count),
        "Rows removed": 0
    },
    {
        "Decision": "Flag missing unit costs",
        "Reason": "Identify incomplete catalog entries instead of imputing arbitrary prices.",
        "Rows affected": int(cost_missing),
        "Rows removed": 0
    },
    {
        "Decision": "Drop unused metadata columns",
        "Reason": "Remove subcategory, supplier, and package_size columns as they are unreliable or redundant.",
        "Rows affected": len(product),
        "Rows removed": 0
    }
])

display(product_decision_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicates,Ensure clean starting baseline for catalog loo...,0,0
1,Standardize raw SKUs,Strip whitespace and uppercase values to allow...,0,0
2,Standardize category labels,Map diverse text strings to official normalize...,32,0
3,Map variants to canonical SKUs,Consolidate multiple variant SKUs (such as pac...,25,0
4,Flag missing unit costs,Identify incomplete catalog entries instead of...,5,0
5,Drop unused metadata columns,"Remove subcategory, supplier, and package_size...",45,0


#Transaction Data Decision Log

In [61]:
step1 = remove_duplicates(transactions.copy())
step2 = clean_text_and_ids(step1.copy())
step2_sku = consolidate_skus(step2.copy())
step3 = clean_timestamps(step2_sku.copy())
step4 = clean_prices(step3.copy())
step5 = clean_categories(step4.copy())
step6 = clean_quantities(step5.copy())

text_changed = (
    step1.fillna("").astype(str)
    != step2.fillna("").astype(str)
).any(axis=1).sum()

decision_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicates",
        "Reason": "Avoid counting the same transaction twice.",
        "Rows affected": len(transactions) - len(step1),
        "Rows removed": len(transactions) - len(step1)
    },
    {
        "Decision": "Clean text and IDs",
        "Reason": "Remove extra spaces, handle blanks, and standardize IDs.",
        "Rows affected": text_changed,
        "Rows removed": 0
    },
    {
        "Decision": "Consolidate SKUs",
        "Reason": "Map messy or redundant raw SKUs to canonical product codes.",
        "Rows affected": (step2_sku["sku"] != step2_sku["canonical_sku"]).sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Convert timestamps to datetime",
        "Reason": "Make different date formats usable for time-based analysis.",
        "Rows affected": step3["timestamp"].notna().sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Keep missing timestamps",
        "Reason": "The correct dates are unknown. exclude these from dated analyses.",
        "Rows affected": step3["timestamp"].isna().sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Convert prices to numeric",
        "Reason": "Remove dollar signs and commas so prices can be used in calculations.",
        "Rows affected": step4["unit_price"].notna().sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize category labels",
        "Reason": "Combine equivalent labels so categories are grouped consistently.",
        "Rows affected": (
            step4["category"] != step5["category"
        ]).sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Replace negative quantities with missing values",
        "Reason": "The correct quantities cannot be verified; exclude them from volume calculations.",
        "Rows affected": step6["quantity_invalid"].sum(),
        "Rows removed": 0
    }
])

display(decision_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicates,Avoid counting the same transaction twice.,30,30
1,Clean text and IDs,"Remove extra spaces, handle blanks, and standa...",5725,0
2,Consolidate SKUs,Map messy or redundant raw SKUs to canonical p...,13991,0
3,Convert timestamps to datetime,Make different date formats usable for time-ba...,24648,0
4,Keep missing timestamps,The correct dates are unknown. exclude these f...,884,0
5,Convert prices to numeric,Remove dollar signs and commas so prices can b...,25532,0
6,Standardize category labels,Combine equivalent labels so categories are gr...,19112,0
7,Replace negative quantities with missing values,The correct quantities cannot be verified; exc...,130,0


#Distribution Data Decision Log

In [62]:
import pandas as pd

# Mirroring the step-by-step cleaning process for distribution centers
dist_step1 = distributions.drop_duplicates().copy()

# Clean extra spaces in text columns
dist_step2 = dist_step1.copy()
distribution_text_cols = ["dc_id", "dc_name", "city", "state"]
for col in distribution_text_cols:
    dist_step2[col] = dist_step2[col].astype(str).str.strip()

# Standardize casing to uppercase
dist_step3 = dist_step2.copy()
dist_step3["dc_id"] = dist_step3["dc_id"].str.upper()
dist_step3["state"] = dist_step3["state"].str.upper()

# Flag approximate capacities before stripping special characters
dist_step4 = dist_step3.copy()
dist_step4["surge_capacity_approximate"] = (
    dist_step4["surge_capacity"].astype(str).str.contains("~")
)

# Strip special characters and convert to numeric
dist_step5 = dist_step4.copy()
dist_step5["surge_capacity"] = pd.to_numeric(
    dist_step5["surge_capacity"]
    .astype(str)
    .str.replace("~", "", regex=False)
    .str.strip()
)

# Dynamically calculate rows affected for text changes
spaces_changed = pd.Series(False, index=distributions.index)
for col in distribution_text_cols:
    spaces_changed |= distributions[col].ne(distributions[col].str.strip())

capitalization_changed = pd.Series(False, index=distributions.index)
for col in ["dc_id", "state"]:
    stripped = distributions[col].str.strip()
    capitalization_changed |= stripped.ne(stripped.str.upper())

distribution_decision_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicates",
        "Reason": "Avoid repeated distribution center records.",
        "Rows affected": len(distributions) - len(dist_step1),
        "Rows removed": len(distributions) - len(dist_step1)
    },
    {
        "Decision": "Remove extra spaces",
        "Reason": "Keep text formatting consistent across keys.",
        "Rows affected": int(spaces_changed.sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize ID and state capitalization",
        "Reason": "Use consistent uppercase labels for clean queries.",
        "Rows affected": int(capitalization_changed.sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Convert surge capacity to numeric",
        "Reason": "Strip string artifacts to allow capacity calculations.",
        "Rows affected": len(dist_step5),
        "Rows removed": 0
    },
    {
        "Decision": "Preserve approximate capacity with a flag",
        "Reason": "Record that Miami's capacity is approximate after removing ~.",
        "Rows affected": int(dist_step5["surge_capacity_approximate"].sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Keep missing shipment cost",
        "Reason": "Birmingham's actual cost is unknown.",
        "Rows affected": int(dist_step5["cost_per_shipment"].isna().sum()),
        "Rows removed": 0
    }
])

display(distribution_decision_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicates,Avoid repeated distribution center records.,0,0
1,Remove extra spaces,Keep text formatting consistent across keys.,0,0
2,Standardize ID and state capitalization,Use consistent uppercase labels for clean quer...,0,0
3,Convert surge capacity to numeric,Strip string artifacts to allow capacity calcu...,5,0
4,Preserve approximate capacity with a flag,Record that Miami's capacity is approximate af...,1,0
5,Keep missing shipment cost,Birmingham's actual cost is unknown.,1,0


#Location Data Decision Log

In [63]:
import pandas as pd

# Step 1: Initialize raw copy and remove exact duplicates (if any exist originally)
loc_step1 = locations.drop_duplicates().copy()

# Step 2: Clean text columns (strip whitespace, standardize state and region labels)
loc_step2 = clean_location_text(loc_step1.copy())

# Step 3: Standardize store IDs (match transaction FL-XXX format)
loc_step3 = clean_location_ids(loc_step2.copy())

# Step 4: Clean types (convert ZIP to text, cast coordinates, and flag missing)
loc_step4 = clean_location_types(loc_step3.copy())

# Step 5: Final deduplication on standardized IDs
loc_step5 = loc_step4.drop_duplicates().reset_index(drop=True)

# --- Dynamic Metric Calculations ---
# Calculate rows affected by text trimming and value replacements (state/region normalization)
text_changed = (
    loc_step1.fillna("").astype(str)
    != loc_step2.fillna("").astype(str)
).any(axis=1).sum()

# Calculate rows affected by ID standardization
id_changed = (
    loc_step2["store_id"].astype(str)
    != loc_step3["store_id"].astype(str)
).sum()

location_decision_log = pd.DataFrame([
    {
        "Decision": "Remove initial exact duplicates",
        "Reason": "Ensure a clean baseline before applying standardization.",
        "Rows affected": len(locations) - len(loc_step1),
        "Rows removed": len(locations) - len(loc_step1)
    },
    {
        "Decision": "Clean text, state, and region labels",
        "Reason": "Strip extra spaces and normalize state (FL) and region (Title Case) text formatting.",
        "Rows affected": int(text_changed),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize store IDs",
        "Reason": "Format store IDs consistently to FL-XXX to successfully match transaction records.",
        "Rows affected": int(id_changed),
        "Rows removed": 0
    },
    {
        "Decision": "Convert ZIP codes to text",
        "Reason": "Cast ZIP codes to string to preserve leading zeros as spatial identifiers instead of numeric values.",
        "Rows affected": len(loc_step4),
        "Rows removed": 0
    },
    {
        "Decision": "Remove duplicate stores after standardization",
        "Reason": "Consolidate duplicate records (Orlando/Kissimmee) that became identical after ID standardization.",
        "Rows affected": len(loc_step4) - len(loc_step5),
        "Rows removed": len(loc_step4) - len(loc_step5)
    },
    {
        "Decision": "Keep and flag missing coordinates",
        "Reason": "Retain store records (Tallahassee) without coordinate data and flag them for geographic safety.",
        "Rows affected": int(loc_step5["coordinates_missing"].sum()),
        "Rows removed": 0
    }
])

display(location_decision_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove initial exact duplicates,Ensure a clean baseline before applying standa...,0,0
1,"Clean text, state, and region labels",Strip extra spaces and normalize state (FL) an...,5,0
2,Standardize store IDs,Format store IDs consistently to FL-XXX to suc...,2,0
3,Convert ZIP codes to text,Cast ZIP codes to string to preserve leading z...,14,0
4,Remove duplicate stores after standardization,Consolidate duplicate records (Orlando/Kissimm...,2,2
5,Keep and flag missing coordinates,Retain store records (Tallahassee) without coo...,1,0


#Inventory and Sales Decision Log

In [95]:
# Find where in the cleaned inventory data there is an NaN in canonical_sku
unmapped_inventory = inventory_clean[inventory_clean["canonical_sku"].isna()]

print(f"Total rows in inventory with missing canonical_sku: {len(unmapped_inventory)}")
print("\nUnique unmapped SKUs and their occurrence counts:")
print(unmapped_inventory["sku"].value_counts())

print("\nFirst 10 rows of matching records:")
display(unmapped_inventory.head(10)[["date", "store_id", "sku", "units_on_hand", "units_on_order", "canonical_sku"]])


Total rows in inventory with missing canonical_sku: 355

Unique unmapped SKUs and their occurrence counts:
sku
UNKNOWN-99     91
LEGACY-PT      90
OLD-BATT-50    90
MISC-001       84
Name: count, dtype: int64

First 10 rows of matching records:


,date,store_id,sku,units_on_hand,units_on_order,canonical_sku
13,2024-09-12,FL-105,LEGACY-PT,176,46.0,NaN
31,2024-09-12,FL-330,LEGACY-PT,115,47.0,NaN
48,2024-09-12,FL-287,LEGACY-PT,190,22.0,NaN
67,2024-09-12,FL-156,MISC-001,195,NaN,NaN
96,2024-09-12,FL-619,OLD-BATT-50,82,44.0,NaN
97,2024-09-12,FL-619,MISC-001,180,28.0,NaN
103,2024-09-12,FL-619,UNKNOWN-99,151,8.0,NaN
112,2024-09-12,FL-445,UNKNOWN-99,144,10.0,NaN
117,2024-09-12,FL-445,LEGACY-PT,69,58.0,NaN
126,2024-09-12,FL-302,OLD-BATT-50,76,72.0,NaN


In [64]:
import pandas as pd

# =====================================================================
# 1. INVENTORY LEVELS DECISION LOG PIPELINE
# =====================================================================
inv_step1 = inventory.drop_duplicates().copy()

# Standardize Store IDs and SKUs
inv_step2 = inv_step1.copy()
inv_step2["store_id"] = (
    inv_step2["store_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace("_", "-", regex=False)
    .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
)
inv_step2["sku"] = inv_step2["sku"].astype("string").str.strip().str.upper()

# Convert Date
inv_step3 = inv_step2.copy()
inv_step3["date"] = pd.to_datetime(
    inv_step3["date"].astype("string").str.strip(),
    format="mixed",
    errors="coerce"
).dt.normalize()

# Map Variants to Canonical Product Codes
inv_step4 = inv_step3.merge(sku_map, on="sku", how="left")

# --- Dynamic Inventory Metrics ---
inv_text_changed = (
    inv_step1.fillna("").astype(str)
    != inv_step2.fillna("").astype(str)
).any(axis=1).sum()

inventory_decision_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicate rows",
        "Reason": "Ensure a clean baseline before applying standardization steps.",
        "Rows affected": len(inventory) - len(inv_step1),
        "Rows removed": len(inventory) - len(inv_step1)
    },
    {
        "Decision": "Standardize store IDs and SKUs",
        "Reason": "Clean text whitespaces, uppercase SKUs, and format store IDs consistently to FL-XXX.",
        "Rows affected": int(inv_text_changed),
        "Rows removed": 0
    },
    {
        "Decision": "Parse inventory dates",
        "Reason": "Convert string-based dates to datetime64 for time-series compatibility.",
        "Rows affected": int(inv_step3["date"].notna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Flag unmapped legacy SKUs",
        "Reason": "Flag legacy items (like UNKNOWN-99 and LEGACY-PT) without removing them.",
        "Rows affected": int(inv_step4["canonical_sku"].isna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Flag missing units-on-order values",
        "Reason": "Keep incomplete records but mark null orders for visibility.",
        "Rows affected": int(inv_step4["units_on_order"].isna().sum()),
        "Rows removed": 0
    }
])

# =====================================================================
# 2. DAILY SALES SUMMARY DECISION LOG PIPELINE
# =====================================================================
sales_step1 = daily_sales.drop_duplicates().copy()

# Standardize Store IDs
sales_step2 = sales_step1.copy()
sales_step2['store_id'] = (
    sales_step2['store_id']
    .str.strip()
    .str.upper()
    .str.replace('_', '-', regex=False)
    .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
)

# Parse Date
sales_step3 = sales_step2.copy()
sales_step3['date'] = pd.to_datetime(
    sales_step3['date'].astype(str).str.strip(),
    format='mixed',
    errors='coerce'
)

# Standardize Categories
sales_step4 = sales_step3.copy()
sales_step4['category'] = (
    sales_step4['category']
    .str.strip()
    .str.lower()
    .replace(category_mapping)
    .str.title()
)

# --- Dynamic Sales Metrics ---
sales_text_changed = (
    sales_step1["store_id"].str.strip() != sales_step2["store_id"]
).sum()

category_labels_changed = (
    sales_step3["category"] != sales_step4["category"]
).sum()

daily_sales_decision_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicate reports",
        "Reason": "Eliminate redundant daily reports.",
        "Rows affected": len(daily_sales) - len(sales_step1),
        "Rows removed": len(daily_sales) - len(sales_step1)
    },
    {
        "Decision": "Standardize store IDs",
        "Reason": "Align store IDs with standard corporate layout FL-XXX.",
        "Rows affected": int(sales_text_changed),
        "Rows removed": 0
    },
    {
        "Decision": "Parse mixed date formats",
        "Reason": "Handle alternative formats (like 20240911) safely.",
        "Rows affected": int(sales_step3["date"].notna().sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize daily sales category labels",
        "Reason": "Normalize varying category labels to corporate schema.",
        "Rows affected": int(category_labels_changed),
        "Rows removed": 0
    },
    {
        "Decision": "Preserve and flag negative revenues",
        "Reason": "Flag anomalous negative values instead of modifying raw transaction sums.",
        "Rows affected": int((sales_step4["total_revenue"] < 0).sum()),
        "Rows removed": 0
    },
    {
        "Decision": "Preserve and flag fractional units sold",
        "Reason": "Document fractional unit artifacts instead of rounded assumptions.",
        "Rows affected": int((sales_step4["total_units"].mod(1) != 0).sum()),
        "Rows removed": 0
    }
])

print("=== INVENTORY LEVELS DECISION LOG ===")
display(inventory_decision_log)
print("\n=== DAILY SALES SUMMARY DECISION LOG ===")
display(daily_sales_decision_log)

=== INVENTORY LEVELS DECISION LOG ===


,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicate rows,Ensure a clean baseline before applying standa...,0,0
1,Standardize store IDs and SKUs,"Clean text whitespaces, uppercase SKUs, and fo...",0,0
2,Parse inventory dates,Convert string-based dates to datetime64 for t...,4320,0
3,Flag unmapped legacy SKUs,Flag legacy items (like UNKNOWN-99 and LEGACY-...,355,0
4,Flag missing units-on-order values,Keep incomplete records but mark null orders f...,626,0



=== DAILY SALES SUMMARY DECISION LOG ===


,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicate reports,Eliminate redundant daily reports.,0,0
1,Standardize store IDs,Align store IDs with standard corporate layout...,0,0
2,Parse mixed date formats,Handle alternative formats (like 20240911) saf...,3600,0
3,Standardize daily sales category labels,Normalize varying category labels to corporate...,0,0
4,Preserve and flag negative revenues,Flag anomalous negative values instead of modi...,34,0
5,Preserve and flag fractional units sold,Document fractional unit artifacts instead of ...,76,0


# Weather API Integration

In [65]:
import pandas as pd, numpy as np, requests, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,wind_speed_10m_max,pressure_msl_mean',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)           # backoff: 2s, 4s
            continue
        if r.status_code in RETRY_STATUSES:    # 429 or 5xx: wait and retry
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                   # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])   # string -> datetime on the API side
        frame['source'] = 'open-meteo'
        return frame

# Get real weather data directly from the API
weather = daily_weather("2024-09-01", "2024-09-30")

print("source:", weather["source"].iloc[0])
print(weather.dtypes)
display(weather.head())

# Orlando sales by date and product category
sales_daily = (
    transactions_clean
    .loc[lambda d: d["store_id"] == "FL-239"]
    .dropna(subset=["timestamp", "quantity"])
    .assign(date=lambda d: d["timestamp"].dt.normalize())
    .groupby(["date", "category_clean"], as_index=False)
    .agg(total_units=("quantity", "sum"))
)

# Attach Orlando weather to each date/category
merged = sales_daily.merge(
    weather,
    on="date",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check that the join preserves rows and sales totals
assert len(merged) == len(sales_daily), "Row count changed"

assert (
    merged["total_units"].sum()
    == sales_daily["total_units"].sum()
), "Sales total changed"

print(merged["_merge"].value_counts())

print(
    "Sales days with no weather:",
    merged.loc[
        merged["_merge"] == "left_only", "date"
    ].dt.date.unique()
)

display(merged.isnull().sum())
display(merged.head())

source: open-meteo
date                  datetime64[ns]
precipitation_sum            float64
wind_speed_10m_max           float64
pressure_msl_mean            float64
source                        object
dtype: object


,date,precipitation_sum,wind_speed_10m_max,pressure_msl_mean,source
0,2024-09-01,3.0,18.8,1014.1,open-meteo
1,2024-09-02,12.4,13.5,1014.1,open-meteo
2,2024-09-03,24.8,22.8,1015.6,open-meteo
3,2024-09-04,22.1,15.5,1017.7,open-meteo
4,2024-09-05,22.7,15.2,1017.3,open-meteo


_merge
both          300
left_only       0
right_only      0
Name: count, dtype: int64
Sales days with no weather: []


,0
date,0
category_clean,0
total_units,0
precipitation_sum,0
wind_speed_10m_max,0
pressure_msl_mean,0
source,0
_merge,0


,date,category_clean,total_units,precipitation_sum,wind_speed_10m_max,pressure_msl_mean,source,_merge
0,2024-09-01,Alcohol,5,3.0,18.8,1014.1,open-meteo,both
1,2024-09-01,Baby,4,3.0,18.8,1014.1,open-meteo,both
2,2024-09-01,Breakfast,6,3.0,18.8,1014.1,open-meteo,both
3,2024-09-01,Electronics,3,3.0,18.8,1014.1,open-meteo,both
4,2024-09-01,Emergency,31,3.0,18.8,1014.1,open-meteo,both


In [66]:
orlando_orders_daily = (
    transactions_clean
    .loc[lambda d: d["store_id"] == "FL-239"]
    .dropna(subset=["timestamp"])
    .assign(date=lambda d: d["timestamp"].dt.normalize())
    .groupby("date", as_index=False)
    .agg(orders=("transaction_id", "nunique"))
)

orders_weather = orlando_orders_daily.merge(
    weather,
    on="date",
    how="left",
    validate="one_to_one"
)

display(orders_weather.head())

,date,orders,precipitation_sum,wind_speed_10m_max,pressure_msl_mean,source
0,2024-09-01,42,3.0,18.8,1014.1,open-meteo
1,2024-09-02,46,12.4,13.5,1014.1,open-meteo
2,2024-09-03,48,24.8,22.8,1015.6,open-meteo
3,2024-09-04,47,22.1,15.5,1017.7,open-meteo
4,2024-09-05,60,22.7,15.2,1017.3,open-meteo


In [67]:
display(
    merged[
        [
            "date",
            "category_clean",
            "total_units",
            "precipitation_sum",
            "wind_speed_10m_max",
            "pressure_msl_mean"
        ]
    ].head()
)

,date,category_clean,total_units,precipitation_sum,wind_speed_10m_max,pressure_msl_mean
0,2024-09-01,Alcohol,5,3.0,18.8,1014.1
1,2024-09-01,Baby,4,3.0,18.8,1014.1
2,2024-09-01,Breakfast,6,3.0,18.8,1014.1
3,2024-09-01,Electronics,3,3.0,18.8,1014.1
4,2024-09-01,Emergency,31,3.0,18.8,1014.1


We retrieved historical weather data for Orlando for September 1–30, 2024, using the Open-Meteo API: https://archive-api.open-meteo.com/v1/archive. The data include daily precipitation, maximum wind speed, and mean sea-level pressure. We requested weather in the America/New_York timezone and assumed Orlando sales timestamps use the same local time. We joined weather to Orlando store sales (FL-239) by date. This analysis covers Orlando, not all Florida stores.

# Question 1: Demand Surge Identification - Which product categories experienced the greatest sales surge in the 7 days before hurricane landfall compared to the prior baseline period? Quantify the percentage increase and visualize the daily trend for the top 5 categories.

# Question 2: The Pop-Tarts Effect (SKU Consolidation) - After standardizing all Pop-Tart SKU variants into a single product, what is the true daily sales volume? How does the uncleaned (fragmented) view compare to the cleaned (consolidated) view? What business decisions would differ between the two views?

# Question 3: Weather Correlation Analysis - Using weather data pulled from your chosen free API, how do specific weather indicators (wind speed, precipitation, barometric pressure) correlate with daily sales volume across categories? Is there a detectable lag between weather severity escalation and purchasing spikes?

# Question 4: Store-Level Geographic Patterns - Do all Florida store locations experience the same surge patterns, or do stores closer to the predicted hurricane path show earlier and larger spikes? Identify any stores that appear to be outliers.

# Question 5: Signal vs. Artifact — The Board Games Investigation - The board games/entertainment category shows a +250% sales increase. Using the SQLite inventory and sales database, investigate whether this is a genuine demand signal or a data artifact (e.g., SKU miscoding, category lumping, inventory reporting lag). Present your evidence and make a recommendation: should Walmart stock extra board games for the next storm?



#Reflections Questions


1. Data Quality Impact

Describe a specific data quality issue you encountered in the supplied data. How did your cleaning decision change the outcome of your analysis? What would have happened if you had not cuaght it?

2. ETL Trade-Offs

You made choices about how to standardize SKUs, handle missing timestamps, and join weather data. Pick one decision and explain: what alternative approach could you have taken, and how might it have changed your results?

3. Pipeline Trust

The Walmart CDO argued that the biggest risk is not the model -- it is the pipeline. Based on your experience in this project, do you agree or disagree? What was the most fragile part of your data pipeline?

4. Business vs. Data

For Question 5 (board games), your team had to make a recommendation with imperfect data. How did you weigh statistical evidence against business risk? Would you have decided differently with more time or more data?

5. Team Collaboration

How did your team divide the work? What would you do differently if you had another week? What was the most valuable skill each team member contributed?